<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/SectorLeadershipModern.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install --upgrade yfinance

In [ ]:

import yfinance as yf
print(yf.__version__)
import pandas as pd
#import pandas_ta as ta
import numpy as np
from datetime import datetime
import time
print("Libraries Installed!")

0.2.65
Libraries Installed!


In [ ]:
df = yf.download('ORCL', period="2y", interval="1wk",auto_adjust=True,group_by='column')
# Flatten MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df2.head()
#df.columns

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2023-07-03,111.740997,113.076705,111.623999,111.965236,7000800
2023-07-10,116.284348,116.966829,110.590542,111.419263,41141700
2023-07-17,115.107452,118.737274,112.739753,116.281516,45321900
2023-07-24,113.483322,115.978217,112.514719,115.391185,35380100
2023-07-31,111.966820,115.489018,111.790709,113.913811,29108200


In [ ]:
def calculate_adx(df, period=14):
    """
    Adds +DI, -DI, and ADX columns to the DataFrame.

    Parameters:
    - df: DataFrame with 'High', 'Low', and 'Close' columns
    - period: Lookback period for calculation (default is 14)

    Returns:
    - df: DataFrame with additional columns: '+DI', '-DI', 'ADX'
    """
    try:
      high = df['High']
      low = df['Low']
      close = df['Close']
      # Calculate directional movements
      up_move = high.diff()
      down_move = -low.diff()

      plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
      minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

      # Calculate True Range (TR)
      tr1 = high - low
      tr2 = (high - close.shift()).abs()
      tr3 = (low - close.shift()).abs()
      tr = pd.DataFrame({'tr1': tr1, 'tr2': tr2, 'tr3': tr3}).max(axis=1)

      # Smooth TR, +DM, and -DM
      atr = tr.rolling(window=period).sum()
      plus_dm_smoothed = pd.Series(plus_dm).rolling(window=period).sum()
      minus_dm_smoothed = pd.Series(minus_dm).rolling(window=period).sum()

      # Calculate +DI and -DI
      plus_di = 100 * (plus_dm_smoothed / atr)
      minus_di = 100 * (minus_dm_smoothed / atr)

      # Calculate DX and ADX
      dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
      adx = dx.rolling(window=period).mean()

      # Add results to DataFrame
      df['+DI'] = plus_di
      df['-DI'] = minus_di
      df['ADX'] = adx

      return df
    except Exception as e:
      print("Error calculating +DMI, -DMI and ADX")

In [ ]:
df = calculate_adx(df, period=14)
df.tail()

Price,Close,High,Low,Open,Volume,+DI,-DI,ADX
Date,,,,,,,,
2025-06-09,215.220001,216.600006,173.789993,174.860001,147045900,NaN,NaN,NaN
2025-06-16,205.169998,215.880005,204.639999,213.199997,76419000,NaN,NaN,NaN
2025-06-23,210.240005,216.929993,202.539993,205.509995,68912200,NaN,NaN,NaN
2025-06-30,237.320007,237.990005,216.309998,226.500000,89574700,NaN,NaN,NaN
2025-07-07,232.259995,235.250000,229.500000,235.110001,16573800,NaN,NaN,NaN


In [ ]:
high = df['High']
low = df['Low']
close = df['Close']

# Calculate directional movements
up_move = high.diff()
down_move = -low.diff()

plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)

# Calculate True Range (TR)
tr1 = high - low
tr2 = (high - close.shift()).abs()
tr3 = (low - close.shift()).abs()
tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

 # Smooth TR, +DM, and -DM using Wilder’s smoothing
atr = tr.rolling(window=14).sum()
plus_dm_series = pd.Series(plus_dm.ravel(), index=df.index).astype(float)
minus_dm_series = pd.Series(minus_dm.ravel(), index=df.index).astype(float)

plus_dm_smoothed = plus_dm_series.rolling(window=14).sum()
minus_dm_smoothed = minus_dm_series.rolling(window=14).sum()

# Directional Indicators
plus_di = 100 * (plus_dm_smoothed / atr)
minus_di = 100 * (minus_dm_smoothed / atr)

# DX and ADX
dx = 100 * (np.abs(plus_di - minus_di) / (plus_di + minus_di))
adx = dx.rolling(window=14).mean()

# Add results to original DataFrame
df['+DI'] = plus_di
df['-DI'] = minus_di
df['ADX'] = adx
df['di_flag'] = df['+DI'] > df['-DI']
df['di_flag'] = df['di_flag'].astype(int)
df['adx_indicator'] = np.where(df['ADX'] > 25, 1, 0)
df['adx_signal'] = df['adx_indicator'] * df['di_flag']
#df2 = calculate_adx(df, period=14)
df.tail()


Price,Close,High,Low,Open,Volume,+DI,-DI,ADX,di_flag,adx_indicator,adx_signal
Date,,,,,,,,,,,
2025-06-09,215.220001,216.600006,173.789993,174.860001,147045900,37.908978,21.424255,34.130713,1,1,1
2025-06-16,205.169998,215.880005,204.639999,213.199997,76419000,38.999089,17.014701,34.574728,1,1,1
2025-06-23,210.240005,216.929993,202.539993,205.509995,68912200,36.260641,17.402508,35.401750,1,1,1
2025-06-30,237.320007,237.990005,216.309998,226.500000,89574700,43.732993,12.534138,37.481746,1,1,1
2025-07-07,232.259995,235.250000,229.500000,235.110001,16573800,46.341557,7.741595,40.157650,1,1,1
